In [2]:
#imports
import pandas as pd
import yfinance as yf
import requests


/Users/goncalomaio/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
#get tickers
headers = {"User-Agent": "Mozilla/5.0"}
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

tables = pd.read_html(requests.get(url, headers=headers).content)
tickers = tables[0]["Symbol"].str.replace(".", "-", regex=False).tolist()
print(f"Found {len(tickers)} tickers.")
tickers[:10]

Found 503 tickers.


['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']

In [ ]:
#fetch 
PERIOD = '1y'

data = yf.download(tickers, period=PERIOD, auto_adjust=True, group_by='ticker', progress=False)

raw = {}
for ticker in tickers:
    try:
        df = data[ticker][['Open', 'High', 'Low', 'Close', 'Volume']].copy()
        df.columns = ['open', 'high', 'low', 'close', 'volume']
        df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
        df.index.name = 'date'
        df = df.dropna(how='all')
        if not df.empty:
            raw[ticker] = df
    except Exception as e:
        print(f'Error {ticker}: {e}')

print(f'Downloaded: {len(raw)} tickers')

50/503 done...
100/503 done...
150/503 done...
200/503 done...
250/503 done...
300/503 done...
350/503 done...
400/503 done...
450/503 done...
500/503 done...

Downloaded: 503 tickers


In [5]:
#clean data
def clean(df):
    df = df[~df.index.duplicated(keep='last')]
    df = df.dropna(subset=['open', 'close'])
    df = df[(df['open'] > 0) & (df['close'] > 0) & (df['volume'] > 0)]
    return df

raw = {ticker: clean(df) for ticker, df in raw.items()}
print('cleaning done.')

cleaning done.


In [6]:
#liq filter
def passes_liquidity(df, min_dv=10_000_000):
    avg_dv = (df['close'] * df['volume']).tail(20).mean()
    return avg_dv >= min_dv

universe = {t: df for t, df in raw.items() if passes_liquidity(df)}
print(f'{len(universe)} tickers')

503 tickers


In [7]:

summary = []
for ticker, df in universe.items():
    if len(df) < 130:
        continue
    r = (df['open'] / df['close'].shift(1)) - 1
    summary.append({
        'ticker': ticker,
        'avg_overnight_%': round(r.mean() * 100, 4),
        'std_%': round(r.std() * 100, 4),
        'rows': len(df)
    })

summary_df = pd.DataFrame(summary).sort_values('avg_overnight_%', ascending=False)
summary_df.head(10)

,ticker,avg_overnight_%,std_%,rows
405,SNDK,0.5977,3.3131,251
489,WDC,0.4064,2.1619,251
314,MU,0.3766,2.4043,251
408,STX,0.3444,1.9344,251
207,FCX,0.3192,2.0682,251
119,FIX,0.2977,2.2471,251
438,TER,0.2890,2.4529,251
6,AMD,0.2859,3.3298,251
73,AVGO,0.2556,2.1397,251
212,GEV,0.2506,1.9703,251


In [10]:
#signal generator
import json
import os
from datetime import date
from datetime import datetime

signals = []

for ticker, df in universe.items():
    if len(df) < 130:
        continue
    df['overnight_return'] = (df['open'] / df['close'].shift(1)) - 1
    cum_overnight = (1 + df['overnight_return']).tail(126).prod() - 1
    signals.append({'ticker': ticker, 'cum_overnight_126d': cum_overnight})

signals_df = pd.DataFrame(signals).sort_values('cum_overnight_126d', ascending=False)

# top 10
top10 = signals_df.head(10)['ticker'].tolist()
print("Top 10 stocks para comprar hoje:")
print(top10)

# save JSON
os.makedirs('signals', exist_ok=True)

filename = f"signals/signals_{datetime.now().strftime('%Y%m%d_%H%M')}.json"

with open(filename, 'w') as f:
    json.dump(output, f, indent=2)

print("\nsignals.json gravado.")

Top 10 stocks para comprar hoje:
['SNDK', 'MU', 'WDC', 'STX', 'FCX', 'TER', 'AVGO', 'LRCX', 'GEV', 'CIEN']

signals.json gravado.
